# Macro Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option('display.max_columns', None)

In [2]:
data_path = os.path.join("..", 'data')
macro_raw = pd.read_csv(os.path.join(data_path, 'macro_data.csv'), index_col=0)

In [3]:
macro_raw.head()

,industrial_production,real_person_income,unemployment_rate,initial_jobless_claims,continuing_claims,cpi,oil_price,vix,credit_spread,baa_aaa_spread,yield_curve_slope,fed_funds_rate,consumer_sentiment,housing_starts,m2_money_supply,dollar_index
1990-01-31,61.7290,7184.9,5.4,345000.0,2316000.0,127.5,22.69,25.36,1.68,0.96,0.15,8.23,93.0,1551.0,3166.8,113.174548
1990-02-28,62.2896,7210.5,5.3,350000.0,2347000.0,128.0,21.55,21.99,1.68,0.94,0.08,8.24,89.5,1437.0,3179.2,113.226110
1990-03-31,62.5999,7216.8,5.2,346000.0,2365000.0,128.6,20.34,19.73,1.58,0.84,0.01,8.28,91.3,1289.0,3190.1,115.508660
1990-04-30,62.4359,7262.7,5.4,363000.0,2403000.0,128.9,18.50,19.52,1.50,0.91,0.08,8.26,93.9,1248.0,3201.6,115.502904
1990-05-31,62.6258,7252.9,5.4,359000.0,2431000.0,129.1,17.47,17.37,1.71,0.94,0.10,8.18,90.6,1212.0,3200.6,113.922498


In [4]:
df = macro_raw.copy()

In [5]:
def adf_kpss_verdict(series: pd.Series, regression="c", alpha=0.05) -> dict:
    """Return both p-values and a joint verdict."""
    s = series.dropna()
    adf_p  = adfuller(s, autolag="AIC")[1]
    kpss_p = kpss(s, regression=regression, nlags="auto")[1]

    if adf_p < alpha and kpss_p > alpha:
        verdict = "stationary"
    elif adf_p >= alpha and kpss_p <= alpha:
        verdict = "unit root"
    else:
        verdict = "inconclusive"
    return {"ADF p": adf_p, "KPSS p": kpss_p, "verdict": verdict}


level_results = pd.DataFrame(
    {code: adf_kpss_verdict(df[code]) for code in df.columns}
).T.round(4)
print("\nLevel-series stationarity tests")
print(level_results)


Level-series stationarity tests
                           ADF p    KPSS p       verdict
industrial_production   0.259264      0.01     unit root
real_person_income      0.988585      0.01     unit root
unemployment_rate       0.072481       0.1  inconclusive
initial_jobless_claims       0.0       0.1    stationary
continuing_claims        0.00008       0.1    stationary
cpi                     0.997855      0.01     unit root
oil_price               0.120168      0.01     unit root
vix                     0.003496       0.1    stationary
credit_spread           0.004411  0.059955    stationary
baa_aaa_spread          0.000443       0.1    stationary
yield_curve_slope       0.007215       0.1    stationary
fed_funds_rate          0.011643      0.01  inconclusive
consumer_sentiment      0.423366      0.01     unit root
housing_starts          0.544406  0.065251  inconclusive
m2_money_supply         0.995871      0.01     unit root
dollar_index            0.326185      0.01     unit roo

In [6]:
TRANSFORM = {
    # Real activity
    'industrial_production':   'dlog',
    'real_person_income':      'dlog',

    # Labour market
    'unemployment_rate':       'diff',
    'initial_jobless_claims':  'dlog',
    'continuing_claims':       'dlog',   # NEW: labour market depth

    # Inflation
    'cpi':                     'dlog',

    # Energy / commodities
    'oil_price':               'dlog',

    # Financial conditions
    'vix':                     'level',
    'credit_spread':           'level',
    'high_yield_spread':       'level',   # NEW: broader credit conditions
    'yield_curve_slope':       'level',

    # Monetary policy
    'fed_funds_rate':          'diff',

    # Sentiment & housing
    'consumer_sentiment':      'diff',
    'housing_starts':          'dlog',

    # Money supply
    'm2_money_supply':         'dlog',

    # Dollar index
    'dollar_index':            'dlog',   # NEW: exchange rate / USD strength
}

def transform(series: pd.Series, kind: str) -> pd.Series:
    if kind == 'level':
        return series
    if kind == 'diff':
        return series.diff()
    if kind == 'dlog':
        return np.log(series).diff()
    raise ValueError(f'Unknown transform: {kind}')

# Apply only to columns present in df (handles missing series gracefully)
macro_stationary = pd.DataFrame(
    {col: transform(df[col], TRANSFORM[col])
     for col in df.columns if col in TRANSFORM}
).dropna()

print(f'Transformed panel shape: {macro_stationary.shape}')
print(f'Indicators: {list(macro_stationary.columns)}')
macro_stationary.head()


Transformed panel shape: (431, 15)
Indicators: ['industrial_production', 'real_person_income', 'unemployment_rate', 'initial_jobless_claims', 'continuing_claims', 'cpi', 'oil_price', 'vix', 'credit_spread', 'yield_curve_slope', 'fed_funds_rate', 'consumer_sentiment', 'housing_starts', 'm2_money_supply', 'dollar_index']


,industrial_production,real_person_income,unemployment_rate,initial_jobless_claims,continuing_claims,cpi,oil_price,vix,credit_spread,yield_curve_slope,fed_funds_rate,consumer_sentiment,housing_starts,m2_money_supply,dollar_index
1990-02-28,0.009041,0.003557,-0.1,0.014389,0.013296,0.003914,-0.051548,21.99,1.68,0.08,0.01,-3.5,-0.076342,0.003908,0.000455
1990-03-31,0.004969,0.000873,-0.1,-0.011494,0.007640,0.004677,-0.057786,19.73,1.58,0.01,0.04,1.8,-0.108691,0.003423,0.019959
1990-04-30,-0.002623,0.006340,0.2,0.047964,0.015940,0.002330,-0.094819,19.52,1.50,0.08,-0.02,2.6,-0.032324,0.003598,-0.000050
1990-05-31,0.003037,-0.001350,0.0,-0.011080,0.011585,0.001550,-0.057286,17.37,1.71,0.10,-0.08,-3.3,-0.029270,-0.000312,-0.013777
1990-06-30,0.003386,0.001213,-0.2,0.013831,0.004515,0.006178,-0.024335,15.50,1.79,0.19,0.11,-2.3,-0.029303,0.004085,-0.000131


In [7]:
trans_results = pd.DataFrame(
    {col: adf_kpss_verdict(macro_stationary[col]) for col in macro_stationary.columns}
).T.round(4)

print("\nPost-transformation stationarity tests")
print(trans_results)


Post-transformation stationarity tests
                           ADF p    KPSS p     verdict
industrial_production        0.0  0.073781  stationary
real_person_income           0.0       0.1  stationary
unemployment_rate            0.0       0.1  stationary
initial_jobless_claims       0.0       0.1  stationary
continuing_claims            0.0       0.1  stationary
cpi                     0.001373       0.1  stationary
oil_price                    0.0       0.1  stationary
vix                     0.003015       0.1  stationary
credit_spread           0.004089  0.061956  stationary
yield_curve_slope       0.006535       0.1  stationary
fed_funds_rate          0.000225       0.1  stationary
consumer_sentiment           0.0       0.1  stationary
housing_starts               0.0       0.1  stationary
m2_money_supply         0.000957       0.1  stationary
dollar_index                 0.0       0.1  stationary


In [8]:
comparison = pd.concat(
    [
        level_results.rename(columns={
            "ADF p": "ADF_level", "KPSS p": "KPSS_level", "verdict": "verdict_level"
        }),
        trans_results.rename(columns={
            "ADF p": "ADF_trans", "KPSS p": "KPSS_trans", "verdict": "verdict_trans"
        }),
    ],
    axis=1
)
print("\nLevel vs transformed comparison")
comparison


Level vs transformed comparison


,ADF_level,KPSS_level,verdict_level,ADF_trans,KPSS_trans,verdict_trans
industrial_production,0.259264,0.01,unit root,0.0,0.073781,stationary
real_person_income,0.988585,0.01,unit root,0.0,0.1,stationary
unemployment_rate,0.072481,0.1,inconclusive,0.0,0.1,stationary
initial_jobless_claims,0.0,0.1,stationary,0.0,0.1,stationary
continuing_claims,0.00008,0.1,stationary,0.0,0.1,stationary
cpi,0.997855,0.01,unit root,0.001373,0.1,stationary
oil_price,0.120168,0.01,unit root,0.0,0.1,stationary
vix,0.003496,0.1,stationary,0.003015,0.1,stationary
credit_spread,0.004411,0.059955,stationary,0.004089,0.061956,stationary
baa_aaa_spread,0.000443,0.1,stationary,NaN,NaN,NaN


In [9]:
scaler = StandardScaler()
macro_scaled_array = scaler.fit_transform(macro_stationary)

macro_scaled = pd.DataFrame(
    macro_scaled_array,
    index=macro_stationary.index,
    columns=macro_stationary.columns,
)

print("Mean of each standardised variable (should be ~0):")
print(macro_scaled.mean().round(4))
print("\nStandard deviation of each standardised variable (should be ~1):")
print(macro_scaled.std().round(4))
print(f"\nFinal panel shape: {macro_scaled.shape}")
print(f"Date range: {macro_scaled.index.min()} to {macro_scaled.index.max()}")

Mean of each standardised variable (should be ~0):
industrial_production     0.0
real_person_income       -0.0
unemployment_rate        -0.0
initial_jobless_claims   -0.0
continuing_claims        -0.0
cpi                       0.0
oil_price                 0.0
vix                       0.0
credit_spread            -0.0
yield_curve_slope         0.0
fed_funds_rate           -0.0
consumer_sentiment        0.0
housing_starts            0.0
m2_money_supply           0.0
dollar_index              0.0
dtype: float64

Standard deviation of each standardised variable (should be ~1):
industrial_production     1.0012
real_person_income        1.0012
unemployment_rate         1.0012
initial_jobless_claims    1.0012
continuing_claims         1.0012
cpi                       1.0012
oil_price                 1.0012
vix                       1.0012
credit_spread             1.0012
yield_curve_slope         1.0012
fed_funds_rate            1.0012
consumer_sentiment        1.0012
housing_starts        

### New indicator transformation notes

**Continuing claims** (`CCSA`): log-differenced like initial claims — captures changes in the stock of unemployed workers claiming benefits.  Level is non-stationary; growth rate is stationary and interpretable.

**High yield spread** (`BAMLH0A0HYM2`): kept in levels like credit spread and VIX — these are already spread measures (differences of rates) and are stationary in levels.  Captures broader credit market stress beyond the investment-grade BAA spread.

**Dollar index** (`DTWEXBGS`): log-differenced — captures monthly appreciation/depreciation of the USD against a trade-weighted basket of currencies.  Dollar appreciation is associated with tightening global financial conditions and typically headwinds for gold and commodities.

**Note on real interest rate**: real interest rates are not added as a separate series to avoid multicollinearity with FEDFUNDS and CPI, both of which are already in the panel.  PLS/PCA will find the relevant FEDFUNDS-CPI combination automatically.

In [10]:
macro_scaled.to_csv(os.path.join(data_path, 'macro_clean.csv'))

# Asset Data

In [11]:
price_raw = pd.read_csv(os.path.join(data_path, 'market_data.csv'), index_col="Date")

In [12]:
df1 = price_raw.copy()

In [13]:
df1 = df1.pct_change().dropna()

In [14]:
df1.head()

,index_fund,treasury_fund,gold_fund
Date,,,
1990-02-28,0.012747,-0.002577,0.017748
1990-03-31,0.018880,-0.004423,-0.051637
1990-04-30,-0.018122,-0.027425,-0.001374
1990-05-31,0.096928,0.045251,-0.013624
1990-06-30,-0.011840,0.023153,-0.003990


In [15]:
df1.to_csv(os.path.join(data_path, 'market_clean.csv'))